'Задача 1: Создание таблицы с первичным ключом
Создайте новую таблицу Airlines (авиакомпании), которая будет хранить информацию об авиакомпаниях. Записать данные туда рандомные.
Таблица должна содержать следующие поля:
- airline_id (идентификатор авиакомпании, целое число, уникальное значение).
- airline_name (название авиакомпании, строка).
- country (страна регистрации авиакомпании, строка).
- Обеспечьте уникальность airline_id !'


In [ ]:

create table bookings.Airlines (
		airline_id integer primary key,
		airline_name varchar(100) not null,
		country varchar(100) not null
		)
		

insert into bookings.Airlines (airline_id, airline_name, country) VALUES
(1, 'Lufthansa', 'Germany'),
(2, 'Air France', 'France'),
(3, 'British Airways', 'United Kingdom'),
(4, 'Emirates', 'United Arab Emirates'),
(5, 'Qatar Airways', 'Qatar'),
(6, 'Singapore Airlines', 'Singapore'),
(7, 'Delta Air Lines', 'USA'),
(8, 'American Airlines', 'USA'),
(9, 'United Airlines', 'USA')

'Задача 2: Добавление внешнего ключа
Добавьте поле airline_id в таблицу Flights, 
чтобы связать каждый 
рейс с авиакомпанией. 
Создайте внешний ключ на это поле, ссылающийся 
на таблицу Airlines.'


In [ ]:

alter table bookings.flights 
add column airline_id Integer

alter table bookings.flights 
add constraint f_key_airlines
foreign key (airline_id)
references bookings.airlines (airline_id)

create or replace function 
random_airline_id()
returns integer as 
$$
declare 
	result integer;
begin
	select airline_id into result
	from bookings.Airlines
	order by random()
	limit 1;
	return result;
end;
$$ language plpgsql;


update bookings.flights  
set airline_id = random_airline_id()


'Задача 3: Частичный индекс для отмененных рейсов
Добавьте поле is_cancelled (логическое значение) 
в таблицу Flights. 
Создайте частичный индекс для ускорения поиска
только отмененных рейсов. 
Сравните результаты до и после создания индекса'



'Задача 4: Составной индекс для поиска билетов
Создайте составной индекс для ускорения поиска 
билетов (Tickets) по имени пассажира и номеру билета. 
Сравните 
результаты до и после создания индекса.'


In [ ]:
explain analyze

select * from bookings.tickets t 
where t.ticket_no = '0005435516610' or t.passenger_name = 'IRINA ZHUKOVA'


--Planning Time: 0.148 ms
--Execution Time: 89.906 ms
-- 475 M



In [ ]:
create index idx_tickets_ticket_no_passenger_name
	on bookings.tickets(passenger_name, ticket_no)


--Planning Time: 0.392 ms
--Execution Time: 1.044 ms
-- 618m
	
-- объем увеличился на 30%
	


'Задача 6*: Оптимизация поиска рейсов по нескольким полям
В таблице flights хранится информация о рейсах. 
Вы часто выполняете запрос на поиск рейсов по аэропорту 
вылета и дате вылета. Создайте составной индекс 
для ускорения этого запроса и сравните результаты'


In [ ]:
explain analyze
select * from bookings.flights f 
where extract(month from f.scheduled_departure) = 2 
			and f.departure_airport = 'OVB'

--Planning Time: 0.184 ms
--Execution Time: 19.210 ms
-- 68 M

In [ ]:
create index idx_flights_scheduled_departure_departure_airport
	on bookings.flights(departure_airport, scheduled_departure)
	

--Planning Time: 0.148 ms
--Execution Time: 4.018 ms	
-- 74 m
-- объем увеличился на 6%


'Оптимизация JOIN-запроса между рейсами и аэропортами
В таблице Flights хранится информация о рейсах, а в 
таблице Airports — информация об аэропортах.
 Вы часто выполняете запрос
 на получение всех рейсов из определенного аэропорта:'

In [ ]:
EXPLAIN ANALYZE
SELECT f.flight_id, f.flight_no, a.airport_name
FROM bookings.flights f
JOIN bookings.airports a ON f.departure_airport = a.airport_code
WHERE a.airport_name = 'Домодедово';


--Planning Time: 0.245 ms
--Execution Time: 24.765 ms
-- объем таблицы flights 68 М
-- объем таблицы airports 64 k 

create index idx_flights_departure_airport
	on bookings.flights(departure_airport)

--Planning Time: 0.320 ms
--Execution Time: 7.325 ms
-- объем таблицы flights 69 М

In [ ]:
create index idx_airports_airport_name
	on bookings.airports(airport_name)

--Planning Time: 0.447 ms
--Execution Time: 7.772 ms

-- при составном индексе по полям airport_code и  airport_name
-- время запроса не меняется 
-- также не меняется время выполнения запроса при частичном индексе по полю airport_code или airport_name
-- оптимальным идексом будет - частичный индекс по полю departure_airport